# One Forecast, Two Plans — Assignment

**Week 7 · Assignment 7** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026
**Points:** 100 &nbsp;·&nbsp; **Due:** *End of Week 7*

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week7/assignments/week07_planning_quantile.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Continues Lesson 8 ("Probabilistic Forecast + Prediction Intervals") —
same dataset, same ARIMA model, a different decision.*

## Setup

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

from gbus564_coursework.utils.course_data import load_aus_retail
from gbus564_coursework.utils.assignment_checks import check_close, check_equal, check_in_range

## Before You Start

- **Format — same shape every week:** **A. Extend** (40 pts, scaffolded
  fill-ins on the week's mechanic) → **B. Decide** (30 pts, one modeling
  decision with a number and a justification) → **C. Recommend** (30 pts, a
  stakeholder-facing writeup). Edit only the `# YOUR CODE HERE` blocks and
  the *\<placeholder\>* prose — everything else is given. This week Part A
  leaves more of the mechanic to you than earlier weeks did.
- **Self-checks are feedback, not the grade.** The check cells after each
  fill-in are checkpoints: green output means you're on track.
- **Submission:** *Restart & Run All*, then download the executed `.ipynb`
  and submit that file — outputs must be present in what you turn in.
- **AI use is expected** (see syllabus). Parts B and C are graded on
  judgment against *your own* results, so ground your answer in the numbers
  your own run produced.

## Summary of Answers

*Fill this in last, after finishing Parts B and C below.*

| Part | Your Answer |
| --- | --- |
| **B — Decide** | _Your answer here._ |
| **C — Recommend** | _Your answer here._ |

## 1. Business Problem

Lesson 8 put an 80% interval around ARIMA's next-month forecast, built
from the model's past misses, and flagged that choosing which edge to plan
around is a cost question, not a statistics one. Lesson 7's pinball loss
slides gave the rule: plan to the percentile **cost of running short ÷
(cost of running short + cost of running over)**.

The same turnover forecast feeds two teams whose misses cost opposite
things. For **staffing**, running short is the expensive miss; for **fresh
stock**, running over is. Operations would like one planning line for both,
to keep things simple.

**Business question:** should both teams plan to the same line — and if
not, which percentile of the forecast should each plan around, and what
does splitting them save?

**Why it matters:** planning too high wastes labor and spoils food;
planning too low turns customers away. Which mistake is more expensive
depends on the team, so the right margin does too.

## 2. Data — Part A: Extend

*Same series and the same three time blocks as Lesson 8 — given below.
Your fill-in is the table statsforecast needs.*

In [ ]:
retail = load_aus_retail()
industry = retail[retail["Industry"] == "Cafes, restaurants and takeaway food services"]
series = industry.groupby("Month")["Turnover"].sum().loc["2004-01-01":"2018-12-01"]
series.index = pd.to_datetime(series.index)

CALIB_START = pd.Timestamp("2013-05-01")
TEST_START = pd.Timestamp("2016-03-01")
n_calib = len(series.loc[CALIB_START:TEST_START - pd.offsets.MonthBegin(1)])
n_test = len(series.loc[TEST_START:])
print(f"{len(series)} months; calibration {n_calib}, test {n_test}")

In [ ]:
# TODO: build `long`, the three-column table statsforecast expects
# (Lessons 6 and 8).
# YOUR CODE HERE
long.head()

In [ ]:
check_equal(list(long.columns), ["unique_id", "ds", "y"], "long has exactly unique_id, ds, y")
check_equal(len(long), len(series), "one row per month")

## 3. Model — Part A: Extend

*Same model and rules as Lesson 8: AutoARIMA fit once on the training
months, then one-month-ahead forecasts through the calibration and test
months, never refit. Collect the calibration misses that size the
margin.*

In [ ]:
%pip install -q statsforecast
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

# TODO: produce Lesson 8's one-month-ahead forecasts through the calibration
# and test months (fit once, never refit), split them at TEST_START into
# `calib` and `test` (keeping statsforecast's "y" and "AutoARIMA" columns),
# and compute:
#   calib_misses -- calibration actual minus forecast, as a NumPy array
#   pred_test    -- the test-month forecasts, as a NumPy array
#   y_test       -- the test-month actuals, as a NumPy array
sf = StatsForecast(models=[AutoARIMA(season_length=12)], freq="MS")
# YOUR CODE HERE

print(f"Calibration misses: {len(calib_misses)}, test forecasts: {len(pred_test)}")

In [ ]:
check_equal(len(calib_misses), n_calib, "one miss per calibration month")
check_equal(len(pred_test), n_test, "one forecast per test month")
check_in_range(np.mean(np.abs(calib_misses)), 10, 150, "calibration misses are a plausible size ($M)")
check_close(calib_misses[0], calib["y"].iloc[0] - calib["AutoARIMA"].iloc[0],
            "misses are actual minus forecast, not the reverse")

## 4. Evaluate — Part A: Extend

*Score the point forecast against Seasonal Naive, as in Lessons 6 and 8.*

In [ ]:
from sklearn.metrics import mean_absolute_error

seasonal_naive_test = series.shift(12).loc[TEST_START:].values
seasonal_naive_mae = mean_absolute_error(y_test, seasonal_naive_test)

# TODO: point_mae -- MAE of pred_test against y_test.
# YOUR CODE HERE

print(f"Point forecast MAE: {point_mae:.1f} (Seasonal Naive: {seasonal_naive_mae:.1f})")

In [ ]:
check_equal(point_mae < seasonal_naive_mae, True, "ARIMA beats the seasonal-naive baseline (matches Lesson 8)")
check_in_range(point_mae, 0, seasonal_naive_mae, "point_mae is a valid, non-trivial MAE")

## 5. Interpret — Part B: Decide

One forecast, two teams, opposite costs:

- **Staffing** — running **short** (the actual lands *above* the plan: a
  rush with too few people on shift) costs **3 times as much** per dollar
  of miss as running **over** (idle staff hours).
- **Fresh stock** — running **over** (the actual lands *below* the plan:
  food thrown out at closing) costs **3 times as much** per dollar of miss
  as running **short** (an item sold out early).

Operations proposes **one planning line for both teams**. Decide whether
to accept it — and if not, which percentile each team should plan to.

*For each team: apply Lesson 7's rule to get its percentile, take that
percentile of the calibration misses as its margin, add the margin to
`pred_test` to get its plan, and check what share of test months came in
at or below the plan — the one-line version of coverage.*

In [ ]:
costs = {"Staffing":    {"short": 3, "over": 1},
         "Fresh stock": {"short": 1, "over": 3}}
percentile, margin, plan, at_or_below = {}, {}, {}, {}

# TODO: for each team in `costs`, fill in the four dicts above, keyed by team:
#   percentile  -- Lesson 7's rule, from that team's short and over costs
#   margin      -- that percentile of calib_misses, as a float
#   plan        -- pred_test plus the margin
#   at_or_below -- share of test months whose actual came in at or below the plan
# YOUR CODE HERE

for team in costs:
    print(f"{team:12s} percentile {percentile[team]:.0%}   margin {margin[team]:+6.1f}   "
          f"test months at or below the plan: {at_or_below[team]:.0%}")

In [ ]:
for team, c in costs.items():
    check_in_range(percentile[team], 0.0, 1.0, f"{team}: percentile is a valid level")
    check_close(margin[team], np.quantile(calib_misses, percentile[team]),
                f"{team}: margin is that percentile of calib_misses")
    check_in_range(at_or_below[team], 0.0, 1.0, f"{team}: share at or below is a valid fraction")

*Given: what each option would have cost over the 34 test months, each
team paying under its own costs (each \$1 of its cheaper miss counted as
\$1, each \$1 of its expensive miss as \$3). This is Lesson 7's "scoring a
decision" bullet: a team's monthly cost is the pinball loss at its own
percentile, times (3 + 1) = 4. The two single-line options are the ones
operations would most likely pick: the point forecast, or Lesson 8's upper
edge (the 90th percentile of the same calibration misses).*

In [ ]:
def avg_monthly_cost(line, team):
    short = np.maximum(y_test - line, 0)       # actual above the line: ran short
    over = np.maximum(line - y_test, 0)        # actual below the line: overstocked
    return np.mean(costs[team]["short"] * short + costs[team]["over"] * over)

upper_edge = pred_test + np.quantile(calib_misses, 0.9)
options = {
    "Two lines (each team's own percentile)": plan,
    "One line: point forecast": {team: pred_test for team in costs},
    "One line: Lesson 8's upper edge (90th)": {team: upper_edge for team in costs},
}
table = pd.DataFrame({name: {team: avg_monthly_cost(lines[team], team) for team in costs}
                      for name, lines in options.items()}).T
table["Total"] = table.sum(axis=1)
print("Average monthly cost on the 34 test months ($M):")
print(table.round(1).to_string())

**Your answer:** *\<accept or reject the single line; each team's
percentile and margin\>*

**Justification (3–5 sentences):** _Your answer here._

## 6. Recommend — Part C: Recommend

*Stakeholder-facing. Write for the operations lead who proposed one
planning line — reference your own Part B numbers.*

**Recommendation:** _Your answer here._

**What would change this recommendation:** _Your answer here._

**Limits:** _Your answer here._

## Rubric

*Same four criteria, same point split, every week — 100 pts total.*

| Criterion | Points | What it measures |
| --- | --- | --- |
| **Correctness** | 40 | Part A's checkpoints pass — the pipeline runs and produces the right shapes/values. |
| **Decision Quality** | 20 | Part B's call on the single line follows from the stated costs, and each team's percentile and margin are correct and on the right side of the forecast. |
| **Justification** | 20 | Part B's reasoning (10 pts) engages with your own numbers; Part C's recommendation (10 pts) follows logically from your own Part B answer. |
| **Communication** | 20 | Part C reads as a stakeholder memo — clear, actionable, and honest about limits. |